# SD card files: load, concatenate and post

Smart Citizen Kits write their readings to the SD card, one file per day (and `.01`, `.02`... files when they restart). This notebook joins them, renames the columns to the names scdata uses, and posts them to the Smart Citizen API.

Posting needs a token of the owner of the device: set `SC_BEARER` in the environment, or in the `.env` file in `~/.cache/scdata`.

In [ ]:
import logging

import scdata as sc
import smartcitizen_connector
from scdata.tools.custom_logger import set_logger_level

# Only errors: 'WARNING', 'INFO' or 'DEBUG' show what scdata and smartcitizen-connector do
set_logger_level('ERROR')
logging.getLogger('smartcitizen_connector').setLevel('ERROR')

sc.__version__

## The files

Place the files of each device in a folder named with its id, for example in `config.paths['raw']`:

```
~/.cache/scdata/raw
├── 13318
│   ├── 20-12-10.01
│   ├── 20-12-10.CSV
│   ├── ...
│   └── 20-12-14.CSV
└── 13319
    ├── 20-12-08.CSV
    └── ...
```

Each file has four header rows: names, units, descriptions and sensor ids. To run this example, we split the example CSV of the repository (it has the same format) as a kit that restarted: a day file and its `.01` file.

In [ ]:
import os
from os.path import join
from scdata._config import config

config.paths['raw']

In [ ]:
folder = join(config.paths['raw'], 'example')
os.makedirs(folder, exist_ok=True)

with open('../../scdata/tools/interim/example.csv') as file:
    lines = file.read().splitlines()
header, rows = lines[:4], lines[4:]

day = rows[0][2:10]
half = len(rows) // 2
for name, part in ((f'{day}.CSV', rows[:half]), (f'{day}.01', rows[half:])):
    with open(join(folder, name), 'w') as file:
        file.write('\n'.join(header + part) + '\n')

sorted(os.listdir(folder)), header[3]

## Join the files

`sdcard_concat` joins the files of a folder, removes repeated readings and, with `output`, saves the result in the folder (`CONCAT.CSV` by default; `''` only returns the data).

With `rename=True`, the columns get the names scdata uses, by the sensor id in the fourth header row (e.g. `PM_1`, id 89, becomes `PMS5003_PM_1`). Columns without id keep their name. Units are not converted.

In [ ]:
from scdata.io import sdcard_concat

data = sdcard_concat(folder, output='', timezone='Europe/Madrid')
list(data.columns)

In [ ]:
data = sdcard_concat(folder, output='CONCAT.CSV', timezone='Europe/Madrid', rename=True)
list(data.columns)

In [ ]:
data.head()

The output file keeps the four header rows, with the new names:

In [ ]:
with open(join(folder, 'CONCAT.CSV')) as file:
    print(''.join(file.readlines()[:5]))

## Post to the Smart Citizen API

Load the joined data in the device it belongs to, and post it. With `dry_run=True`, the payload is printed and nothing is posted: set it to `False` to post (the device must be yours).

In [ ]:
device = sc.Device(blueprint='sc_air', params=sc.APIParams(id=16784))

device.data = data
device.handler.data = device.data

await device.post(columns='sensors', dry_run=True)

## Several devices

The same for each folder of `config.paths['raw']`:

In [ ]:
devices = {}
for name in sorted(os.listdir(config.paths['raw'])):
    path = join(config.paths['raw'], name)
    if not name.isdigit() or not os.path.isdir(path):
        continue
    device = sc.Device(blueprint='sc_air', params=sc.APIParams(id=int(name)))
    device.data = sdcard_concat(path, rename=True)
    device.handler.data = device.data
    devices[name] = device

for name, device in devices.items():
    print(name, device.data.shape)
    # await device.post(columns='sensors', dry_run=True)